In [1]:
import pandas as pd
import numpy as np
import torch

processor = torch.load("processor.pt")
df = processor.df


/tmp/ipykernel_2759/777429502.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  processor = torch.load("processor.pt")


In [5]:
(df["watch_ratio"]>1).mean()

np.float64(0.22588248765642951)

In [17]:
print(df.loc[df["like"] == 1, "click"].mean())    # 应该非常接近1.0
print(df.loc[df["follow"] == 1, "click"].mean())   # 同上

0.718823412032544
0.7316396683485291


In [11]:
s = pd.Series([1, 2, 3, 4])
s.shift(1).cumsum()

0    NaN
1    1.0
2    3.0
3    6.0
dtype: float64

In [66]:
import pandas as pd

pd.set_option('display.max_rows', None)      # 显示所有行
pd.set_option('display.max_columns', None)   # 显示所有列
pd.set_option('display.width', None)         # 自动适应宽度
pd.set_option('display.max_colwidth', None)  # 单元格内容不截断
#画用户层级图
user_click_count = df.groupby("user_id")["click"].count().reset_index(name = "click_cnt")
max = user_click_count["click_cnt"].max()
min = user_click_count["click_cnt"].min()

gap = int((max - min) /200)
print(gap)

user_click_count["bin_idx"] = user_click_count["click_cnt"].apply(lambda x: x // gap)

user_click_count = user_click_count.groupby("bin_idx").count()["click_cnt"].reset_index(name = "click_cnt")
user_click_count["bin"] = user_click_count["bin_idx"].apply(lambda x:f"{x *gap}-{(x+1)*gap}")
user_click_count


46


,bin_idx,click_cnt,bin
0,0,7474,0-46
1,1,2955,46-92
2,2,1770,92-138
3,3,1215,138-184
4,4,903,184-230
5,5,693,230-276
6,6,548,276-322
7,7,469,322-368
8,8,364,368-414
9,9,312,414-460


In [11]:
df["time_since_upload"] = np.log(df["timestamp"].to_date() - df["upload_time"])

AttributeError: 'Series' object has no attribute 'to_date'

In [4]:
# 同一个(user_id, item_id)交互对,在train和val里都出现过的比例
train_pairs = set(zip(processor.train_df["user_id"], processor.train_df["item_id"]))
val_pairs = set(zip(processor.val_df["user_id"], processor.val_df["item_id"]))

overlap_pairs = train_pairs & val_pairs
print(f"train中不同的(user,item)交互对: {len(train_pairs)}")
print(f"val中不同的(user,item)交互对: {len(val_pairs)}")
print(f"val里有 {len(overlap_pairs) / len(val_pairs) * 100:.2f}% 的交互对,"
      f"是同一个用户在train阶段就看过的同一个视频")

# 顺手也算一下更宽松的:物料本身(不分用户)在train/val的重合度
train_items = set(processor.train_df["item_id"])
val_items = set(processor.val_df["item_id"])
item_overlap = train_items & val_items
print(f"val里有 {len(item_overlap) / len(val_items) * 100:.2f}% 的物料,"
      f"train阶段就已经出现过(不分用户)")


train中不同的(user,item)交互对: 4321386
val中不同的(user,item)交互对: 1441238
val里有 0.03% 的交互对,是同一个用户在train阶段就看过的同一个视频
val里有 39.77% 的物料,train阶段就已经出现过(不分用户)


In [ ]:
NUM_FEATURES = [
    "log_duration_ms", "log_playing_time", "log_video_age",
    "hour_of_day", "day_of_week", "is_weekend",
    "user_hist_click_rate_by_cat1", "user_hist_like_rate_by_cat1",
    "user_hist_follow_rate_by_cat1", "user_hist_avg_watch_ratio_by_cat1",
    "user_hist_click_cnt_by_author", "user_hist_like_cnt_by_author",
    "user_hist_click_rate_by_hour",
    "user_hist_click_rate_overall", "user_hist_interaction_cnt",
    "item_hist_click_rate", "item_hist_like_rate", "item_hist_avg_watch_ratio",
    "author_hist_click_rate", "author_hist_avg_watch_ratio",
    "user_recent20_click_rate", "user_recent10_avg_watch_ratio_by_cat1",
    "cross_user_interest_x_item_hotness",
]  # 必须和 KuaiSARTorchDataset 里的 numeric_cols 完全一致(顺序也要一致!)

CAT_FEATURES = [
    "author_id", "item_type", "upload_type", 
    "first_level_category_id", "second_level_category_id",
    "third_level_category_id", "fourth_level_category_id"
]  # 必须和 KuaiSARTorchDataset 里的 categorical_cols 完全一致



corr_click = processor.train_df[NUM_FEATURES + ["click"]].corr()["click"].sort_values(key=abs, ascending=False)

print("跟click相关性最高的特征:")
print(corr_click.head(10))

corr_wr = processor.train_df[NUM_FEATURES + ["watch_ratio"]].corr()["watch_ratio"].sort_values(key=abs, ascending=False)
print("\n跟watch_ratio相关性最高的特征:")
print(corr_wr.head(10))

跟click相关性最高的特征:
click                                    1.000000
log_playing_time                         0.630896
user_recent20_click_rate                 0.366307
user_recent10_avg_watch_ratio_by_cat1    0.171029
user_hist_click_cnt_by_author            0.137762
user_hist_interaction_cnt               -0.100415
log_duration_ms                          0.067987
log_video_age                            0.049734
day_of_week                              0.036597
is_weekend                               0.033002
Name: click, dtype: float64

跟watch_ratio相关性最高的特征:
watch_ratio                              1.000000
log_playing_time                         0.485070
log_duration_ms                         -0.355638
user_recent10_avg_watch_ratio_by_cat1    0.227101
user_recent20_click_rate                 0.219678
user_hist_click_cnt_by_author            0.085103
user_hist_interaction_cnt               -0.069483
user_hist_avg_watch_ratio_by_cat1        0.018891
day_of_week                      

In [7]:
df["author_hist_avg_watch_ratio"]

0         -0.085413
1         -0.085413
2         -0.085413
3         -0.085413
4         -0.085413
             ...   
4328078   -0.077322
4328079   -0.084951
4328080   -0.079977
4328081   -0.076458
4328082   -0.085413
Name: author_hist_avg_watch_ratio, Length: 4328083, dtype: float64

In [8]:
df["user_recent10_avg_watch_ratio_by_cat1"]

0         -1.130126
1         -1.130126
2         -1.130126
3         -1.130126
4         -1.130126
             ...   
4328078   -1.064044
4328079   -1.130126
4328080   -1.028468
4328081   -0.716077
4328082   -1.073185
Name: user_recent10_avg_watch_ratio_by_cat1, Length: 4328083, dtype: float64

In [18]:
"""
看watch_ratio的分布:整体形状、log尺度看尾部、train/val/test是否一致、按click分组对比。
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    plt.rcParams["font.sans-serif"] = ["Noto Sans CJK SC", "SimHei", "Microsoft YaHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

processor = torch.load("processor.pt")  # 或者 torch.load("processor.pt"),看你现在怎么拿到processor

train_df = processor.train_df
val_df = processor.val_df
test_df = processor.test_df  # 属性名跟你自己代码里对上,如果叫别的名字改一下

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0][0]
ax.hist(train_df["watch_ratio"], bins=100, color="#2a78d6", alpha=0.8)
ax.set_title("watch_ratio整体分布(train)")
ax.set_xlabel("watch_ratio"); ax.set_ylabel("count")

ax = axes[0][1]
ax.hist(train_df["watch_ratio"], bins=100, color="#2a78d6", alpha=0.8)
ax.set_yscale("log")
ax.set_title("同上,y轴log(看尾部/看有没有在5那里堆积,因为你clip过0-5)")
ax.set_xlabel("watch_ratio"); ax.set_ylabel("count (log)")

ax = axes[1][0]
for name, d, color in [("train", train_df, "#2a78d6"), ("val", val_df, "#eb6834"), ("test", test_df, "#4a3aa7")]:
    ax.hist(d["watch_ratio"], bins=100, density=True, histtype="step", linewidth=1.8, label=name, color=color)
ax.set_title("train/val/test 分布对比(density,看有没有明显漂移)")
ax.set_xlabel("watch_ratio"); ax.legend()

ax = axes[1][1]
for click_val, color, label in [(0, "#898781", "click=0"), (1, "#eb6834", "click=1")]:
    sub = train_df.loc[train_df["click"] == click_val, "watch_ratio"]
    ax.hist(sub, bins=100, density=True, histtype="step", linewidth=1.8, label=f"{label} (n={len(sub)})", color=color)
ax.set_title("watch_ratio按click分组(train)")
ax.set_xlabel("watch_ratio"); ax.legend()

fig.suptitle("watch_ratio 分布诊断", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("watch_ratio_dist.png", dpi=150, bbox_inches="tight")
print("Saved: watch_ratio_dist.png")

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    s = d["watch_ratio"]
    print(f"--- {name} ---")
    print(s.describe())
    print(f"click=0时watch_ratio=0的比例: {(d.loc[d['click']==0,'watch_ratio']==0).mean():.4f}")
    print(f"click=1时watch_ratio=0的比例: {(d.loc[d['click']==1,'watch_ratio']==0).mean():.4f}")
    print()


/tmp/ipykernel_3574/746923788.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  processor = torch.load("processor.pt")  # 或者 torch.load("processor.pt"),看你现在怎么拿到processor


Saved: watch_ratio_dist.png
--- train ---
count    4.328083e+06
mean     5.300845e-01
std      7.105312e-01
min      0.000000e+00
25%      4.820806e-02
50%      2.176372e-01
75%      9.396741e-01
max      5.000000e+00
Name: watch_ratio, dtype: float64
click=0时watch_ratio=0的比例: 0.0666
click=1时watch_ratio=0的比例: 0.0017

--- val ---
count    1.442694e+06
mean     5.547430e-01
std      7.192855e-01
min      0.000000e+00
25%      5.414754e-02
50%      2.485000e-01
75%      9.798234e-01
max      5.000000e+00
Name: watch_ratio, dtype: float64
click=0时watch_ratio=0的比例: 0.0649
click=1时watch_ratio=0的比例: 0.0015

--- test ---
count    1.442695e+06
mean     5.248363e-01
std      7.192644e-01
min      0.000000e+00
25%      4.458979e-02
50%      2.055613e-01
75%      9.280126e-01
max      5.000000e+00
Name: watch_ratio, dtype: float64
click=0时watch_ratio=0的比例: 0.0686
click=1时watch_ratio=0的比例: 0.0015



In [19]:
n_total = len(train_df)
print(f"总行数: {n_total}")
print(f"click=1比例: {(train_df['click']==1).mean():.4f}")
print(f"click=0比例: {(train_df['click']==0).mean():.4f}")

# 问题2: 是不是只有click=1才可能like=1/follow=1
print(f"\nlike=1的行里,click=1的比例: {train_df.loc[train_df['like']==1,'click'].mean():.4f}")
print(f"follow=1的行里,click=1的比例: {train_df.loc[train_df['follow']==1,'click'].mean():.4f}")

# 反过来看转化率视角:click=1之后,有多少比例进一步like/follow了
print(f"\nclick=1的行里,like=1的比例: {train_df.loc[train_df['click']==1,'like'].mean():.4f}")
print(f"click=1的行里,follow=1的比例: {train_df.loc[train_df['click']==1,'follow'].mean():.4f}")
print(f"click=0的行里,like=1的比例: {train_df.loc[train_df['click']==0,'like'].mean():.4f}")
print(f"click=0的行里,follow=1的比例: {train_df.loc[train_df['click']==0,'follow'].mean():.4f}")

# 问题3: 是否存在"四个信号都接近0"的纯滑走记录
pure_zero = train_df[(train_df['click']==0) & (train_df['like']==0) & (train_df['follow']==0)]
print(f"\nclick/like/follow三者全为0的行占比: {len(pure_zero)/n_total:.4f}")
print(pure_zero['watch_ratio'].describe())

truly_skipped = train_df[(train_df['click']==0) & (train_df['like']==0) &
                          (train_df['follow']==0) & (train_df['watch_ratio'] < 0.05)]
print(f"\n三个动作全0且watch_ratio<0.05(疑似真正划走没怎么看)的行占比: {len(truly_skipped)/n_total:.4f}")

# val集重复一遍,对比train/val的构成是否一致
for name, d in [("train", train_df), ("val", val_df)]:
    print(f"\n--- {name} ---")
    print(f"click=1比例: {(d['click']==1).mean():.4f}")
    print(f"like=1的行里click=1比例: {d.loc[d['like']==1,'click'].mean():.4f}")
    print(f"follow=1的行里click=1比例: {d.loc[d['follow']==1,'click'].mean():.4f}")

总行数: 4328083
click=1比例: 0.4981
click=0比例: 0.5019

like=1的行里,click=1的比例: 0.7183
follow=1的行里,click=1的比例: 0.7219

click=1的行里,like=1的比例: 0.0443
click=1的行里,follow=1的比例: 0.0069
click=0的行里,like=1的比例: 0.0173
click=0的行里,follow=1的比例: 0.0027

click/like/follow三者全为0的行占比: 0.4924
count    2.131173e+06
mean     1.730563e-01
std      3.800945e-01
min      0.000000e+00
25%      1.664855e-02
50%      6.314391e-02
75%      1.739552e-01
max      5.000000e+00
Name: watch_ratio, dtype: float64

三个动作全0且watch_ratio<0.05(疑似真正划走没怎么看)的行占比: 0.2218

--- train ---
click=1比例: 0.4981
like=1的行里click=1比例: 0.7183
follow=1的行里click=1比例: 0.7219

--- val ---
click=1比例: 0.5260
like=1的行里click=1比例: 0.7249
follow=1的行里click=1比例: 0.7582


In [22]:
df.columns

Index(['user_id', 'item_id', 'duration_ms', 'playing_time', 'timestamp',
       'forward', 'like', 'follow', 'search_item_related', 'search', 'click',
       'time', 'caption', 'author_id', 'item_type', 'upload_time',
       'upload_type', 'music_id', 'first_level_category_id',
       'first_level_category_name', 'second_level_category_id',
       'second_level_category_name', 'third_level_category_id',
       'third_level_category_name', 'fourth_level_category_id',
       'fourth_level_category_name', 'first_level_category_name_en',
       'second_level_category_name_en', 'third_level_category_name_en',
       'fourth_level_category_name_en', 'watch_ratio', 'log_duration_ms',
       'log_playing_time', 'log_video_age', 'hour_of_day', 'day_of_week',
       'is_weekend', 'user_hist_click_rate_by_cat1',
       'user_hist_like_rate_by_cat1', 'user_hist_follow_rate_by_cat1',
       'user_hist_avg_watch_ratio_by_cat1', 'user_hist_click_cnt_by_author',
       'user_hist_like_cnt_by_author', 

In [23]:
import pandas as pd

# df = processor.train_df  # 或者你存原始合并表的那个df

print("="*60)
print("1. timestamp vs time 到底是不是一回事")
print("="*60)
print(df[["timestamp", "time"]].dtypes)
print(df[["timestamp", "time"]].head(10))
print("两列完全相等的比例:", (df["timestamp"] == df["time"]).mean())
print("timestamp - time 的差值统计(如果是数值型):")
try:
    print((df["timestamp"].astype("int64") - df["time"].astype("int64")).describe())
except Exception as e:
    print("转换失败,可能类型不一样:", e)

print("\n" + "="*60)
print("2. search / search_item_related 是什么样的字段")
print("="*60)
print("search 取值分布:\n", df["search"].value_counts(dropna=False))
print("\nsearch_item_related 取值分布:\n", df["search_item_related"].value_counts(dropna=False))
print("\nsearch=1 的行占比:", (df["search"] == 1).mean() if df["search"].dtype != object else "非数值,看上面value_counts")

print("\n" + "="*60)
print("3. click/like/follow/forward/search 几个行为标签的关系")
print("="*60)
behavior_cols = ["click", "like", "follow", "forward", "search", "search_item_related"]
for col in behavior_cols:
    print(f"{col} 取值分布:")
    print(df[col].value_counts(dropna=False))
    print()

print("like=1时click的分布:\n", df.loc[df["like"] == 1, "click"].value_counts(normalize=True))
print("follow=1时click的分布:\n", df.loc[df["follow"] == 1, "click"].value_counts(normalize=True))
print("forward=1时click的分布:\n", df.loc[df["forward"] == 1, "click"].value_counts(normalize=True))

print("\n" + "="*60)
print("4. 是否有重复的(user_id, item_id, timestamp)组合(判断是不是纯粹一行一次交互)")
print("="*60)
dup = df.duplicated(subset=["user_id", "item_id", "timestamp"], keep=False)
print("重复行占比:", dup.mean())
if dup.sum() > 0:
    print(df.loc[dup].sort_values(["user_id", "item_id", "timestamp"]).head(20))

print("\n" + "="*60)
print("5. 同一个(user_id, item_id)出现多次的情况(用户是否反复刷到同一个视频)")
print("="*60)
pair_counts = df.groupby(["user_id", "item_id"]).size()
print("单个(user,item)组合出现次数分布:\n", pair_counts.value_counts().head(10))
print("最多的一个组合出现了几次:", pair_counts.max())

print("\n" + "="*60)
print("6. duration_ms / playing_time / watch_ratio 三者是否自洽")
print("="*60)
recompute = (df["playing_time"] / df["duration_ms"]).clip(0, 5)
diff = (recompute - df["watch_ratio"]).abs()
print("重新计算watch_ratio和现有列的差值统计:\n", diff.describe())
print("差值>0.01的行占比:", (diff > 0.01).mean())

print("\n" + "="*60)
print("7. 各列缺失值情况")
print("="*60)
print(df[behavior_cols + ["duration_ms", "playing_time", "watch_ratio"]].isna().mean())

1. timestamp vs time 到底是不是一回事
timestamp    datetime64[ms]
time                    str
dtype: object
                timestamp                 time
0 2023-05-22 06:50:00.151  2023-05-22 14:50:00
1 2023-05-22 06:50:00.151  2023-05-22 14:50:00
2 2023-05-22 06:50:00.151  2023-05-22 14:50:00
3 2023-05-22 06:50:00.151  2023-05-22 14:50:00
4 2023-05-22 06:50:00.151  2023-05-22 14:50:00
5 2023-05-22 06:50:00.937  2023-05-22 14:50:00
6 2023-05-22 06:50:00.937  2023-05-22 14:50:00
7 2023-05-22 06:50:00.937  2023-05-22 14:50:00
8 2023-05-22 06:50:00.966  2023-05-22 14:50:00
9 2023-05-22 06:50:00.966  2023-05-22 14:50:00
两列完全相等的比例: 0.0
timestamp - time 的差值统计(如果是数值型):
转换失败,可能类型不一样: invalid literal for int() with base 10: '2023-05-22 14:50:00'

2. search / search_item_related 是什么样的字段
search 取值分布:
 search
0    7182758
1      30714
Name: count, dtype: int64

search_item_related 取值分布:
 search_item_related
0    7213470
1          2
Name: count, dtype: int64

search=1 的行占比: 0.004257866392217229

3. click

In [24]:
# 行级:这一条交互记录里,五个行为信号是不是全是0
all_zero = df[(df["click"]==0) & (df["like"]==0) & (df["follow"]==0) &
              (df["forward"]==0) & (df["search"]==0)]
print(f"click/like/follow/forward/search全为0的行占比: {len(all_zero)/len(df):.4f}")
print(all_zero["watch_ratio"].describe())

# 更严格一点:五个信号全0,而且watch_ratio也很低,算是"真的划走了没什么反应"
truly_no_action = df[(df["click"]==0) & (df["like"]==0) & (df["follow"]==0) &
                      (df["forward"]==0) & (df["search"]==0) & (df["watch_ratio"] < 0.05)]
print(f"\n五个信号全0 且 watch_ratio<0.05 的行占比: {len(truly_no_action)/len(df):.4f}")

# item级:有没有某个物料,所有曝光过它的用户里没有一个人click/like/follow过它(冷门/长尾物料)
item_level = df.groupby("item_id").agg(
    n_interactions=("user_id", "count"),
    total_click=("click", "sum"),
    total_like=("like", "sum"),
    total_follow=("follow", "sum"),
)
zero_action_items = item_level[(item_level["total_click"] == 0) &
                                (item_level["total_like"] == 0) &
                                (item_level["total_follow"] == 0)]
print(f"\n从未被任何用户click/like/follow过的item占比: {len(zero_action_items)/len(item_level):.4f}")
print(f"这些item平均被曝光了几次: {zero_action_items['n_interactions'].mean():.2f}")

click/like/follow/forward/search全为0的行占比: 0.4841
count    3.492327e+06
mean     1.716119e-01
std      3.766151e-01
min      0.000000e+00
25%      1.671777e-02
50%      6.287770e-02
75%      1.735822e-01
max      5.000000e+00
Name: watch_ratio, dtype: float64

五个信号全0 且 watch_ratio<0.05 的行占比: 0.2185

从未被任何用户click/like/follow过的item占比: 0.4432
这些item平均被曝光了几次: 1.24


In [25]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_cjk_font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
try:
    fm.fontManager.addfont(_cjk_font_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=_cjk_font_path).get_name()
except Exception:
    pass
plt.rcParams["axes.unicode_minus"] = False



print("="*60)
print("1. IQR方法定义的离群点(标准统计学定义)")
print("="*60)
q1, q3 = df["watch_ratio"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print(f"Q1={q1:.4f}, Q3={q3:.4f}, IQR={iqr:.4f}")
print(f"上界={upper:.4f}, 下界={max(lower,0):.4f}(watch_ratio非负,下界意义不大)")
outliers_iqr = df[df["watch_ratio"] > upper]
print(f"超过上界的行占比: {len(outliers_iqr)/len(df):.4f}")

print("\n" + "="*60)
print("2. 卡在clip边界(=5)的那批,是不是数据质量问题(比如视频时长异常短)")
print("="*60)
at_clip = df[df["watch_ratio"] >= 4.999]
print(f"watch_ratio卡在5(clip边界)的行占比: {len(at_clip)/len(df):.4f}")
print("这批行的duration_ms分布(跟全体对比):")
print("全体duration_ms:\n", df["duration_ms"].describe())
print("\nclip边界那批的duration_ms:\n", at_clip["duration_ms"].describe())
print("\nclip边界那批的playing_time:\n", at_clip["playing_time"].describe())

print("\n" + "="*60)
print("3. duration_ms极短的视频,是不是watch_ratio异常高发区(重复播放/统计口径问题)")
print("="*60)
short_videos = df[df["duration_ms"] < df["duration_ms"].quantile(0.01)]
print(f"duration_ms在最短1%分位以下的视频,watch_ratio分布:")
print(short_videos["watch_ratio"].describe())
print(f"这批里watch_ratio>=5(clip上限)的比例: {(short_videos['watch_ratio']>=4.999).mean():.4f}")
print(f"全体watch_ratio>=5的比例(对照组): {(df['watch_ratio']>=4.999).mean():.4f}")

print("\n" + "="*60)
print("4. 哪些item整体平均watch_ratio高得异常(可能是刷量/统计异常物料)")
print("="*60)
item_stats = df.groupby("item_id").agg(
    n=("watch_ratio", "count"),
    mean_wr=("watch_ratio", "mean"),
    dur=("duration_ms", "first"),
).query("n >= 20")  # 至少20次曝光才有统计意义,避免小样本噪声
top_items = item_stats.sort_values("mean_wr", ascending=False).head(20)
print("平均watch_ratio最高的20个item(至少20次曝光):")
print(top_items)

# 画图
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].boxplot(df["watch_ratio"], vert=True)
axes[0].set_title("watch_ratio箱线图(看离群点)")
axes[0].set_ylabel("watch_ratio")

sample = df.sample(min(50000, len(df)), random_state=0)
axes[1].scatter(sample["duration_ms"], sample["watch_ratio"], s=2, alpha=0.2, color="#2a78d6")
axes[1].set_xscale("log")
axes[1].set_title("duration_ms vs watch_ratio(抽样5万点,看是否短视频扎堆在高位)")
axes[1].set_xlabel("duration_ms (log)")
axes[1].set_ylabel("watch_ratio")

fig.tight_layout()
fig.savefig("watch_ratio_outliers.png", dpi=150, bbox_inches="tight")
print("\nSaved: watch_ratio_outliers.png")

1. IQR方法定义的离群点(标准统计学定义)
Q1=0.0486, Q3=0.9475, IQR=0.8989
上界=2.2958, 下界=0.0000(watch_ratio非负,下界意义不大)
超过上界的行占比: 0.0258

2. 卡在clip边界(=5)的那批,是不是数据质量问题(比如视频时长异常短)
watch_ratio卡在5(clip边界)的行占比: 0.0048
这批行的duration_ms分布(跟全体对比):
全体duration_ms:
 count    7.213472e+06
mean     9.342445e+04
std      1.339335e+05
min      7.660000e+02
25%      1.453300e+04
50%      3.683300e+04
75%      1.130000e+05
max      1.982924e+07
Name: duration_ms, dtype: float64

clip边界那批的duration_ms:
 count     34818.000000
mean      13410.652795
std       12380.440137
min        1442.000000
25%        7600.000000
50%       10033.000000
75%       14099.250000
max      224166.000000
Name: duration_ms, dtype: float64

clip边界那批的playing_time:
 count    3.481800e+04
mean     1.148708e+05
std      1.156209e+05
min      9.505000e+03
25%      5.247825e+04
50%      7.552000e+04
75%      1.238418e+05
max      1.253849e+06
Name: playing_time, dtype: float64

3. duration_ms极短的视频,是不是watch_ratio异常高发区(重复播放/统计口径问题)
duration_ms在最短1%分位以下的视频

/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 31665 (\N{CJK UNIFIED IDEOGRAPH-7BB1}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 32447 (\N{CJK UNIFIED IDEOGRAPH-7EBF}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 30475 (\N{CJK UNIFIED IDEOGRAPH-770B}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 31163 (\N{CJK UNIFIED IDEOGRAPH-79BB}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 32676 (\N{CJK UNIFIED IDEOGRAPH-7FA4}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
/tmp/ipykernel_3574/1051404828.py:75: UserWarning: Glyph 28857 (\N{CJK UNIFIED IDEOGRAPH-70B9}


Saved: watch_ratio_outliers.png
